**Bicycle Retail Sales Analytics: Executive Performance Dashboard (2016-2018)**

**1. Introduction & Executive Summary**

This case study transforms raw sales data from a bicycle retail company into actionable business intelligence. The project utilizes an integrated workflow of SQL, Google Sheets, and Tableau to deliver interactive dashboards that enable leadership to monitor performance and identify revenue trends.

The analysis covers three years of sales activity (2016-2018), providing insights across geography, product categories, brand performance, and sales representative effectiveness. 

**2. Business Context & Objectives**

*The Challenge*

Company leadership faced a critical bottleneck: traditional static reports were too lengthy and difficult to navigate. Decision-makers needed a self-service solution to explore data and identify trends without requesting new reports for every question.

*Key Strategic Questions*

* What are the overall sales volume trends from 2016 to 2018? 
* Which regions, stores, and brands drive the highest revenue? 
* Who are the most valuable customers and top-performing sales reps? 
* Are there seasonal patterns to inform inventory and staffing?

**3. Methodology: The 5-Step Process**

I followed a structured analytical framework to ensure accuracy and actionability:

* Understand the Problem: Defined KPIs focused on revenue and comparative performance.
* Collect the Data: Extracted data from a relational database using SQL.
* Clean and Prepare: Validated quality, handled missing values, and calculated derived metrics.
* Analyze and Visualize: Developed dashboards in Excel and Tableau.
* Interpret and Communicate: Translated findings into strategic recommendations.

**4. Technical Implementation: SQL Extraction**

The company’s data was stored in a normalized relational database. To create an analytical dataset, I developed a SQL query to join nine related tables across the sales and production schemas.

*SQL Highlights:*

* Data Consolidation: Joined tables for orders, customers, products, categories, brands, stores, and staff.
* Feature Engineering: Used CONCAT to merge names and calculated revenue as quantity * list_price.
* Filtering: Focused specifically on the 2016-2018 timeframe.

**Query 1: Extraction of data**

In [ ]:
--Create two datasets for the two categories
CREATE SCHEMA IF NOT EXISTS `project-f57ec015-00f5-4842-84d.production`;
CREATE SCHEMA IF NOT EXISTS `project-f57ec015-00f5-4842-84d.sales`; 

-- Create the tables for the two datasets
-- 1. Categories Table
CREATE TABLE IF NOT EXISTS `project-f57ec015-00f5-4842-84d.production.categories` (
  category_id INT64 NOT NULL,
  category_name STRING NOT NULL,
  PRIMARY KEY (category_id) NOT ENFORCED
);

-- 2. Brands Table
CREATE TABLE IF NOT EXISTS `project-f57ec015-00f5-4842-84d.production.brands` (
  brand_id INT64 NOT NULL,
  brand_name STRING NOT NULL,
  PRIMARY KEY (brand_id) NOT ENFORCED
);

-- 3. Products Table
CREATE TABLE IF NOT EXISTS `project-f57ec015-00f5-4842-84d.production.products` (
  product_id INT64 NOT NULL,
  product_name STRING NOT NULL,
  brand_id INT64 NOT NULL,
  category_id INT64 NOT NULL,
  model_year INT64 NOT NULL,
  list_price NUMERIC NOT NULL,
  PRIMARY KEY (product_id) NOT ENFORCED,
  FOREIGN KEY (category_id) REFERENCES `project-f57ec015-00f5-4842-84d.production.categories`(category_id) NOT ENFORCED,
  FOREIGN KEY (brand_id) REFERENCES `project-f57ec015-00f5-4842-84d.production.brands`(brand_id) NOT ENFORCED
);

-- 4. Customers Table
CREATE TABLE IF NOT EXISTS `project-f57ec015-00f5-4842-84d.sales.customers` (
  customer_id INT64 NOT NULL,
  first_name STRING NOT NULL,
  last_name STRING NOT NULL,
  phone STRING,
  email STRING NOT NULL,
  street STRING,
  city STRING,
  state STRING,
  zip_code STRING,
  PRIMARY KEY (customer_id) NOT ENFORCED
);

-- 5. Stores Table
CREATE TABLE IF NOT EXISTS `project-f57ec015-00f5-4842-84d.sales.stores` (
  store_id INT64 NOT NULL,
  store_name STRING NOT NULL,
  phone STRING,
  email STRING,
  street STRING,
  city STRING,
  state STRING,
  zip_code STRING,
  PRIMARY KEY (store_id) NOT ENFORCED
);

-- 6. Staffs Table
CREATE TABLE IF NOT EXISTS `project-f57ec015-00f5-4842-84d.sales.staffs` (
  staff_id INT64 NOT NULL,
  first_name STRING NOT NULL,
  last_name STRING NOT NULL,
  email STRING NOT NULL,
  phone STRING,
  active INT64 NOT NULL,
  store_id INT64 NOT NULL,
  manager_id INT64,
  PRIMARY KEY (staff_id) NOT ENFORCED,
  FOREIGN KEY (store_id) REFERENCES `project-f57ec015-00f5-4842-84d.sales.stores`(store_id) NOT ENFORCED
);

-- 7. Orders Table
CREATE TABLE IF NOT EXISTS `project-f57ec015-00f5-4842-84d.sales.orders` (
  order_id INT64 NOT NULL,
  customer_id INT64,
  order_status INT64 NOT NULL, -- 1 = Pending; 2 = Processing; 3 = Rejected; 4 = Completed
  order_date DATE NOT NULL,
  required_date DATE NOT NULL,
  shipped_date DATE,
  store_id INT64 NOT NULL,
  staff_id INT64 NOT NULL,
  PRIMARY KEY (order_id) NOT ENFORCED,
  FOREIGN KEY (customer_id) REFERENCES `project-f57ec015-00f5-4842-84d.sales.customers`(customer_id) NOT ENFORCED,
  FOREIGN KEY (store_id) REFERENCES `project-f57ec015-00f5-4842-84d.sales.stores`(store_id) NOT ENFORCED
);

-- 8. Order Items Table
CREATE TABLE IF NOT EXISTS `project-f57ec015-00f5-4842-84d.sales.order_items` (
  order_id INT64 NOT NULL,
  item_id INT64 NOT NULL,
  product_id INT64 NOT NULL,
  quantity INT64 NOT NULL,
  list_price NUMERIC NOT NULL,
  discount NUMERIC DEFAULT 0,
  PRIMARY KEY (order_id, item_id) NOT ENFORCED,
  FOREIGN KEY (order_id) REFERENCES `project-f57ec015-00f5-4842-84d.sales.orders`(order_id) NOT ENFORCED,
  FOREIGN KEY (product_id) REFERENCES `project-f57ec015-00f5-4842-84d.production.products`(product_id) NOT ENFORCED
);

-- 9. Stocks Table
CREATE TABLE IF NOT EXISTS `project-f57ec015-00f5-4842-84d.production.stocks` (
  store_id INT64 NOT NULL,
  product_id INT64 NOT NULL,
  quantity INT64,
  PRIMARY KEY (store_id, product_id) NOT ENFORCED,
  FOREIGN KEY (store_id) REFERENCES `project-f57ec015-00f5-4842-84d.sales.stores`(store_id) NOT ENFORCED,
  FOREIGN KEY (product_id) REFERENCES `project-f57ec015-00f5-4842-84d.production.products`(product_id) NOT ENFORCED
);

**Query 2: Loadind data into tables**

In [ ]:
--Load data into the created tables
-- 1. Insert Brands
INSERT INTO `project-f57ec015-00f5-4842-84d.production.brands` (brand_id, brand_name) 
VALUES
  (1, 'Electra'),
  (2, 'Haro'),
  (3, 'Heller'),
  (4, 'Pure Cycles'),
  (5, 'Ritchey'),
  (6, 'Strider'),
  (7, 'Sun Bicycles'),
  (8, 'Surly'),
  (9, 'Trek');

-- 2. Insert into production.categories
INSERT INTO `project-f57ec015-00f5-4842-84d.production.categories` (category_id, category_name) 
VALUES
  (1, 'Children Bicycles'),
  (2, 'Comfort Bicycles'),
  (3, 'Cruisers Bicycles'),
  (4, 'Cyclocross Bicycles'),
  (5, 'Electric Bikes'),
  (6, 'Mountain Bikes'),
  (7, 'Road Bikes');

-- 3. Insert into production.products (Partial Sample)
INSERT INTO `project-f57ec015-00f5-4842-84d.production.products` (product_id, product_name, brand_id, category_id, model_year, list_price) 
VALUES
  (1, 'Trek 820 - 2016', 9, 6, 2016, 379.99),
  (2, 'Ritchey Timberwolf Frameset - 2016', 5, 6, 2016, 749.99),
  (3, 'Surly Wednesday Frameset - 2016', 8, 6, 2016, 999.99),
  (4, 'Trek Fuel EX 8 29 - 2016', 9, 6, 2016, 2899.99),
  (5, 'Heller Shagamaw Frame - 2016', 3, 6, 2016, 1320.99),
  (6, 'Surly Ice Cream Truck Frameset - 2016', 8, 6, 2016, 469.99),
  (7, 'Trek Slash 8 27.5 - 2016', 9, 6, 2016, 3999.99),
  (8, 'Trek Remedy 29 Carbon Frameset - 2016', 9, 6, 2016, 1799.99),
  (9, 'Trek Conduit+ - 2016', 9, 5, 2016, 2999.99),
  (10, 'Surly Straggler - 2016', 8, 4, 2016, 1549.00);
-- (Continue with remaining product values from the file)

-- 4. Insert into sales.orders (Sample showing Date conversion)
-- Original format '20180418' converted to '2018-04-18'
INSERT INTO `project-f57ec015-00f5-4842-84d.sales.orders` (order_id, customer_id, order_status, order_date, required_date, shipped_date, store_id, staff_id) 
VALUES
  (1555, 1, 1, '2018-04-18', '2018-04-18', NULL, 2, 7),
  (1556, 4, 2, '2018-04-18', '2018-04-18', NULL, 2, 6),
  (1557, 121, 2, '2018-04-19', '2018-04-19', NULL, 1, 3);
-- (Continue with remaining order values)

-- 5. Insert into sales.order_items (Sample)
INSERT INTO `project-f57ec015-00f5-4842-84d.sales.order_items` (order_id, item_id, product_id, quantity, list_price, discount) 
VALUES
  (232, 3, 26, 2, 599.99, 0.07),
  (232, 4, 8, 2, 1799.99, 0.1),
  (233, 1, 14, 1, 269.99, 0.1),
  (233, 2, 16, 2, 599.99, 0.1);
-- (Continue with remaining order_item values)

-- 6. Insert into production.stocks (Sample)
INSERT INTO `project-f57ec015-00f5-4842-84d.production.stocks` (store_id, product_id, quantity) 
VALUES
  (1, 72, 9),
  (1, 73, 7),
  (1, 74, 9),
  (1, 75, 23);
-- (Continue with remaining stock values)

**Query 3: Creating new cleaned dataset and summary table**

In [ ]:
-- Create a new dataset for the summary table since there are two datasets used
CREATE SCHEMA `project-f57ec015-00f5-4842-84d.BikeStores`;

-- Create the summary table
CREATE TABLE `project-f57ec015-00f5-4842-84d.BikeStores.bike_stores` AS
  SELECT
    ord.order_id,
    CONCAT (cus.first_name, ' ', cus.last_name) AS customers,
    cus.city,
    cus.state,
    ord.order_date,
    SUM(ite.quantity) AS total_units,
    SUM(ite.quantity * ite.list_price) AS revenue,
    pro.product_name,
    cat.category_name,
    sto.store_name,
    CONCAT (sta.first_name, ' ', sta.last_name) AS sales_rep,
  FROM `project-f57ec015-00f5-4842-84d.sales.orders` AS ord
  JOIN `project-f57ec015-00f5-4842-84d.sales.customers` AS cus
    ON ord.customer_id = cus.customer_id
  JOIN `project-f57ec015-00f5-4842-84d.sales.order_items` AS ite
    ON ord.order_id = ite.order_id
  JOIN `project-f57ec015-00f5-4842-84d.production.products` AS pro
    ON ite.product_id = pro.product_id
  JOIN `project-f57ec015-00f5-4842-84d.production.categories` AS cat
    ON pro.category_id = cat.category_id
  JOIN `project-f57ec015-00f5-4842-84d.sales.stores` AS sto
    ON ord.store_id = sto.store_id
  JOIN `project-f57ec015-00f5-4842-84d.sales.staffs` AS sta
    ON ord.staff_id = sta.staff_id
  GROUP BY
    ord.order_id,
    CONCAT (cus.first_name, ' ', cus.last_name),
    cus.city,
    cus.state,
    ord.order_date,
    pro.product_name,
    cat.category_name,
    sto.store_name,
    CONCAT (sta.first_name, ' ', sta.last_name);

**5. Analysis and Visualization (Google Sheets)**

I used Google Sheets to transform the cleaned dataset into an interactive analytical tool. By leveraging pivot tables and slicers, I created a dynamic environment that allows users to filter data by year, state, or store.

The resulting dashboard provides a comprehensive view of business health through the following visualizations:
* Pie Chart: Illustrates each store's contribution to total revenue, aiding in resource allocation decisions.
* Column Chart: Displays annual revenue for 2016, 2017, and 2018 to identify growth trends.
* Stepped Area Chart: Utilized to visualize revenue accumulation and significant shifts in sales volume over the three-year period.
* Line Graph: Shows monthly revenue patterns across 36 months, highlighting seasonality and peaks in demand.
* Geo Graph: Visualizes geographic distribution by state, instantly highlighting top-performing regions like New York and underperforming areas like Texas.

**6. Interactive Visualizations**

Tableau was used for high-level executive presentation, prioritizing clean design and advanced interactivity:

* Geographic Mapping: Visualized state-level performance with color gradients to identify regional opportunities.
* Tree Maps: Efficiently displayed product category revenue without clutter.
* Dynamic Parameters: Implemented a "Top N" filter to allow executives to toggle between viewing the Top 5, 10, or 20 customers

**7. Key Findings & Insights**

* Revenue Concentration: A small group of high-value clients contributes a significant portion of revenue.
* Geographic Variation: New York is the top-performing state, while Texas shows the lowest regional performance.
* Brand Dominance: Trek is the most profitable brand, significantly outperforming competitors.
* Top Talent: Sales representatives Marceline Boyer and Venita Daniel are consistent top performers.

**8. Strategic Recommendations**

* Loyalty Programs: Implement targeted retention strategies for the Top 10 customers.
* Sales Training: Use the methods of top performers (Boyer/Daniel) to train the wider sales team.
* Regional Investigation: Conduct market research in Texas to identify the root causes of underperformance.
* Inventory Optimization: Prioritize Trek inventory levels, especially during peak seasonal months.

**9. Conclusion**

This project demonstrates the full data lifecycle—from raw SQL tables to executive-ready dashboards. By moving from static reports to self-service analytics, the company can now accelerate decision-making and drive sustainable growth.

*Technology Stack: SQL (BigQuery), Google Sheets, Tableau.*

https://public.tableau.com/views/BikeStoreSalesRevenueDashboard/BikeStoreDashboard?:language=en-US&:sid=&:display_count=n&:origin=viz_share_link